## **Adversarial Attack to AI Image compression**

An AI image compression model can be represented as a Variational AutoEncoder which comprises two levels

In [1]:
import os
import socket
import sys
import warnings
import numpy as np
import torch
from PIL import Image
from io import BytesIO
import imageio.v3 as iio
from compressai.zoo import models as compressai_models
from utils import get_device, evaluate_frequency_response2
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
warnings.filterwarnings("ignore", category=UserWarning, module="pytorch_wavelets")

device = get_device()
hostname = socket.gethostname()

/home/nkalmykov/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU 1: Memory Free = 75083 MB, Temperature = 53°C
Selected GPU: 1 (Memory Free: 75083 MB, Temperature: 53°C)
Running task on GPU-1
Device set to: cuda:1


### Load Image and Compression Model

In [2]:
BASE_DIR = "/home/nkalmykov/compressai_project/experiments"

class ImageCodecModel:
    def __init__(self, codec: str, q_level: int):
        self.codec = codec.lower()
        self.q_level = int(q_level)

    @staticmethod
    def _map_q_to_quality(q: int, codec: str) -> int:
        # Map abstract q (1..6 or 1..8) to a codec quality scale (rough heuristic)
        # JPEG/WEBP quality in [10..95]
        max_q = 6
        q = max(1, min(int(q), max_q))
        return int(np.linspace(20, 95, num=max_q)[q-1])

    def __call__(self, x: torch.Tensor):
        # x: [1,3,H,W] in [0,1]
        img = x.squeeze(0).permute(1,2,0).detach().cpu().numpy()
        img_u8 = np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)
        H, W, _ = img_u8.shape
        buf = BytesIO()
        q_val = self._map_q_to_quality(self.q_level, self.codec)

        if self.codec == 'jpeg':
            Image.fromarray(img_u8, mode='RGB').save(buf, format='JPEG', quality=q_val, subsampling=0, optimize=True)
            buf.seek(0)
            out = Image.open(buf).convert('RGB')
            out_u8 = np.array(out)
        elif self.codec == 'webp':
            Image.fromarray(img_u8, mode='RGB').save(buf, format='WEBP', quality=q_val, method=6)
            buf.seek(0)
            out = Image.open(buf).convert('RGB')
            out_u8 = np.array(out)
        elif self.codec == 'jpeg2000':
            # Prefer imagecodecs direct encoder with explicit PSNR and irreversible transform; fallback to imageio
            psnr_list = [20.0, 24.0, 28.0, 32.0, 36.0, 40.0]
            cr_list = [500, 200, 100, 50, 25, 12]
            idx = min(max(self.q_level, 1), 6) - 1
            target_psnr = psnr_list[idx]
            cr = cr_list[idx]
            try:
                import imagecodecs as _ic
                # irreversible=True enables lossy 9/7 wavelet; mct=1 enables RGB transform
                encoded = _ic.jpeg2k_encode(img_u8, psnr=target_psnr, irreversible=True, mct=1)
                if os.environ.get('CODEC_DEBUG', '0') == '1':
                    print(f"[jpeg2000] backend=imagecodecs psnr={target_psnr} cr={cr} bytes={len(encoded)}")
                out_u8 = _ic.jpeg2k_decode(encoded)
            except Exception:
                tmp = np.ascontiguousarray(img_u8)
                with BytesIO() as b2:
                    try:
                        # Try psnr if backend supports, else use cratio/rate
                        iio.imwrite(b2, tmp, extension='.jp2', psnr=target_psnr)
                        backend = 'imageio-psnr'
                    except Exception:
                        try:
                            iio.imwrite(b2, tmp, extension='.jp2', cratio=cr)
                            backend = 'imageio-cratio'
                        except Exception:
                            rate = max(0.05, 8.0 / cr)
                            try:
                                iio.imwrite(b2, tmp, extension='.jp2', rate=rate)
                                backend = 'imageio-rate'
                            except Exception:
                                iio.imwrite(b2, tmp, extension='.jp2')
                                backend = 'imageio-default'
                    size_bytes = len(b2.getvalue())
                    if os.environ.get('CODEC_DEBUG', '0') == '1':
                        print(f"[jpeg2000] backend={backend} psnr={target_psnr} cr={cr} bytes={size_bytes}")
                    b2.seek(0)
                    out_u8 = iio.imread(b2)
            if out_u8.ndim == 2:
                out_u8 = np.stack([out_u8]*3, axis=-1)
        elif self.codec in ('jpegxl',):
            # JPEG XL. Map q -> distance; lower distance == higher quality
            dist_map = [4.0, 3.0, 2.0, 1.5, 1.0, 0.6]
            dist = dist_map[min(max(self.q_level, 1), 6) - 1]
            out_u8 = None
            # 1) Try imagecodecs' JPEG XL bindings
            try:
                import imagecodecs as _ic
                encoded = _ic.jpegxl_encode(img_u8, distance=dist, effort=7)
                if os.environ.get('CODEC_DEBUG', '0') == '1':
                    print(f"[jpegxl] backend=imagecodecs distance={dist} bytes={len(encoded)}")
                out_u8 = _ic.jpegxl_decode(encoded)
            except Exception:
                # 2) Try cjxl/djxl CLI if installed
                try:
                    import subprocess as _sp, tempfile as _tf, os as _os
                    with _tf.TemporaryDirectory() as _td:
                        inp = _os.path.join(_td, 'in.png')
                        jxl = _os.path.join(_td, 'out.jxl')
                        outp = _os.path.join(_td, 'out.png')
                        iio.imwrite(inp, img_u8, extension='.png')
                        _sp.run(['cjxl', inp, jxl, f'--distance={dist}', '--effort=7'], check=True, stdout=_sp.DEVNULL, stderr=_sp.DEVNULL)
                        size_bytes = _os.path.getsize(jxl) if _os.path.exists(jxl) else -1
                        if os.environ.get('CODEC_DEBUG', '0') == '1':
                            print(f"[jpegxl] backend=cjxl distance={dist} bytes={size_bytes}")
                        _sp.run(['djxl', jxl, outp], check=True, stdout=_sp.DEVNULL, stderr=_sp.DEVNULL)
                        out_u8 = iio.imread(outp)
                except Exception:
                    # 3) Fallback to imageio writer with distance (if supported)
                    try:
                        with BytesIO() as b2:
                            iio.imwrite(b2, img_u8, extension='.jxl', distance=dist)
                            size_bytes = len(b2.getvalue())
                            if os.environ.get('CODEC_DEBUG', '0') == '1':
                                print(f"[jpegxl] backend=imageio distance={dist} bytes={size_bytes}")
                            b2.seek(0)
                            out_u8 = iio.imread(b2)
                    except Exception:
                        # Last resort: write without quality control
                        with BytesIO() as b2:
                            iio.imwrite(b2, img_u8, extension='.jxl')
                            size_bytes = len(b2.getvalue())
                            if os.environ.get('CODEC_DEBUG', '0') == '1':
                                print(f"[jpegxl] backend=imageio-default distance={dist} bytes={size_bytes}")
                            b2.seek(0)
                            out_u8 = iio.imread(b2)
            if out_u8.ndim == 2:
                out_u8 = np.stack([out_u8]*3, axis=-1)
        else:
            raise ValueError(f"Unsupported codec: {self.codec}")

        out_f = (out_u8.astype(np.float32) / 255.0)
        x_hat = torch.from_numpy(out_f).permute(2,0,1).unsqueeze(0)
        return {"x_hat": x_hat}


def load_codec_model(codec_name: str, quality: int, device):
    return ImageCodecModel(codec_name, quality)


def load_tcm_model(p, device):
    """Load and configure the TCM model."""
    # Import the TCM model with correct path
    sys.path.append(os.path.join(BASE_DIR, "LIC_TCM-main"))

    # Map p values to checkpoint paths
    checkpoint_map = {
        128: os.path.abspath(os.path.join(BASE_DIR, "LIC_TCM-main/mse_lambda_0.05.pth.tar")),
        64: os.path.abspath(os.path.join(BASE_DIR, "LIC_TCM-main/mse_lambda_0.0025.pth.tar"))
    }

    if p not in checkpoint_map:
        raise ValueError(f"Unsupported p value: {p}. Supported values: {list(checkpoint_map.keys())}")

    # Load checkpoint
    checkpoint_path = checkpoint_map[p]
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Clean state dictionary
    state_dict = {k.replace("module.", ""): v for k, v in checkpoint["state_dict"].items()}

    # Initialize and configure model
    from models.tcm import TCM
    model = TCM(
        config=[2, 2, 2, 2, 2, 2],
        head_dim=[8, 16, 32, 32, 16, 8],
        drop_path_rate=0.0,
        N=p,
        M=320
    ).to(device).eval()

    model.load_state_dict(state_dict)
    model.update()  # Required before compression

    return model

def load_compressai_model(model_name, quality, device):
    """Load and configure a CompressAI model."""
    model_class = compressai_models.get(model_name, None)
    if not model_class:
        raise ValueError(f"Model {model_name} not found in compressai.zoo.models")

    # Clear GPU memory
    torch.cuda.empty_cache()

    # Load model and set to evaluation mode
    model = model_class(quality=quality, pretrained=True).to(device)
    
    # Disable gradients for parameters
    for param in model.parameters():
        param.requires_grad = False

    return model

# Main model loading logic
def load_model(model_name, quality, device, p=128):
    """Load the specified model based on model_name."""
    if model_name == 'tcm':
        return load_tcm_model(p, device)
    else:
        return load_compressai_model(model_name, quality, device)


# Override load_model to route to codecs as well
_prev_load_model = load_model

def load_model(model_name, quality, device, p=128):
    name = model_name.lower()
    if name in {'jpeg','jpegxl','webp'}:
        return load_codec_model(name, quality, device)
    return _prev_load_model(model_name, quality, device, p=p)

In [5]:
# Batch evaluation and saving results
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import idct

# Configure experiment grids
# Base set
_base_models = ['tcm','cheng2020-anchor', 'cheng2020-attn', 'jpeg', 'webp']  #, 'jpegxl'
# Extra CompressAI zoo models to try (work with evaluate_frequency_response2 via model(x)['x_hat'])
_extra_zoo = ['bmshj2018-factorized', 'bmshj2018-hyperprior', 'mbt2018-mean', 'mbt2018']
# Keep only those available in this environment
try:
    _zoo_available = [m for m in _extra_zoo if compressai_models.get(m, None)]
except Exception:
    _zoo_available = []
model_range = _base_models + _zoo_available

quality_range = [1, 2, 3, 4, 5, 6]
size_range = [64, 128, 256, 512, 1024]
# TCM-specific: evaluate only large sizes, no quality; use p in {64,128}
tcm_size_range = [256, 512, 1024]
tcm_p_range = [64, 128]
print('Models to evaluate:', model_range)

# Where to store results
BASE_RESULTS_DIR = Path('results')
BASE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation parameters
NUM_RUNS = 100
SEED = 42
SAVE_CE_WINDOW = 2

all_runs = []

for model_name in model_range:
    # Select size and quality loops depending on model
    if model_name == 'tcm':
        sizes_iter = tcm_size_range
    else:
        sizes_iter = size_range

    for size in sizes_iter:
        print(f"\n=== Model: {model_name} | Size: {size}x{size} ===")
        size_dir = BASE_RESULTS_DIR / model_name / str(size)
        size_dir.mkdir(parents=True, exist_ok=True)

        per_size_rows = []

        # Determine iteration over quality or p
        if model_name == 'tcm':
            qp_iter = tcm_p_range  # we will treat these as 'p' values
        else:
            qp_iter = quality_range

        for qp in qp_iter:
            if model_name == 'tcm':
                p = qp
                print(f"Evaluating p={p}...")
                q_dir = size_dir / f"p_{p}"
                q_dir.mkdir(parents=True, exist_ok=True)
                model = load_model(model_name, quality=None, device=device, p=p)
            else:
                q = qp
                print(f"Evaluating quality {q}...")
                q_dir = size_dir / f"q_{q}"
                q_dir.mkdir(parents=True, exist_ok=True)
                model = load_model(model_name, q, device)

            x_dct_rgb, x_hat, metrics = evaluate_frequency_response2(
                model,
                size=size,
                device=device,
                show_plots=False,
                num_runs=NUM_RUNS,
                show_metric_plots=False,
                seed=SEED,
            )

            # Normalize original and decompressed DCT for saving
            mx, Mx = x_dct_rgb.min(), x_dct_rgb.max()
            denom = (Mx - mx + 1e-9)

            x_dct_norm = np.clip((x_dct_rgb - mx) / denom, 0.0, 1.0).astype(np.float32)
            x_hat_norm = np.clip((x_hat      - mx) / denom, 0.0, 1.0).astype(np.float32)

            # iDCT визуализация
            x_idct = np.zeros_like(x_hat)
            for c in range(3):
                x_idct[..., c] = idct(x_hat[..., c], axis=1, norm='ortho')
            x_idct_disp = np.clip(
                (x_idct - x_idct.min()) / (x_idct.max() - x_idct.min() + 1e-9),
                0.0, 1.0
            ).astype(np.float32)

            plt.imsave(q_dir / 'original_dct_rgb.png',     x_dct_norm)
            plt.imsave(q_dir / 'decompressed_dct_rgb.png', x_hat_norm)
            plt.imsave(q_dir / 'idct_of_decompressed_rgb.png', x_idct_disp)

            # Save metrics plots (2x3 grid)
            indices = metrics['indices']
            leakage = metrics['leakage']
            odr = metrics['odr']
            centroid_shift = metrics['centroid_shift']
            spread = metrics['spread']
            entropy_bits = metrics['entropy'] / np.log(2)
            cum_energy = metrics['cum_energy']

            fig, axs = plt.subplots(2, 3, figsize=(16, 8))
            axs[0, 0].plot(indices, leakage); axs[0, 0].set_title('Leakage L_k'); axs[0, 0].set_xlabel('k')
            axs[0, 1].plot(indices, odr); axs[0, 1].set_title('Off–diagonal ratio ODR_k'); axs[0, 1].set_xlabel('k')
            axs[0, 2].plot(indices, np.abs(centroid_shift)); axs[0, 2].set_title('Centroid shift Δc_k'); axs[0, 2].set_xlabel('k')
            axs[1, 0].plot(indices, spread); axs[1, 0].set_title('Spread s_k'); axs[1, 0].set_xlabel('k')
            axs[1, 1].plot(indices, entropy_bits); axs[1, 1].set_title('Entropy H_k (bits)'); axs[1, 1].set_xlabel('k')
            for w_key in sorted(cum_energy.keys()):
                axs[1, 2].plot(indices, cum_energy[w_key], label=f"w={w_key}")
            axs[1, 2].set_title('Cumulative energy CE_k(w)'); axs[1, 2].set_xlabel('k'); axs[1, 2].set_ylim(-0.05, 1.05); axs[1, 2].legend()
            fig.tight_layout()
            fig.savefig(q_dir / 'metrics_grid.png', dpi=150)
            plt.close(fig)

            # Save R heatmap for reference
            R = metrics['R']
            fig_hm = plt.figure(figsize=(6, 5))
            plt.imshow(R, aspect='auto', origin='lower', cmap='viridis')
            plt.colorbar(label='Normalized power')
            plt.xlabel('input basis k'); plt.ylabel('observed frequency i')
            plt.title('Frequency-response matrix R')
            fig_hm.tight_layout()
            fig_hm.savefig(q_dir / 'R_heatmap.png', dpi=150)
            plt.close(fig_hm)

            # Collect summary numbers for the per-size table
            s = metrics.get('summary', None)
            if s is None:
                # Build a summary here if not present
                ce_w = metrics['cum_energy'].get(SAVE_CE_WINDOW)
                s = {
                    'L_k': float(np.median(1.0 - np.diag(R))),
                    'ODR_k': float(np.median(odr)),
                    '|Delta_c_k|': float(np.median(np.abs(centroid_shift))),
                    's_k': float(np.median(spread)),
                    'H_k_bits': float(np.median(entropy_bits)),
                    f'CE_k(w={SAVE_CE_WINDOW})': float(np.median(ce_w)) if ce_w is not None else np.nan,
                }

            # Build row metadata depending on model
            row = {'Model': model_name, 'Size': f'{size}x{size}'}
            if model_name == 'tcm':
                row['p'] = int(p)
            else:
                row['q'] = int(q)
            row.update(s)

            # Add compact band-wise summaries if available (disabled: we save only median metrics)
            band = None

            per_size_rows.append(row)
            all_runs.append(row)

        # Save per-size CSV summary (merge: append new, overwrite existing rows by key)
        df_new = pd.DataFrame(per_size_rows)
        base_cols = ['Model','Size','L_k','ODR_k','|Delta_c_k|','s_k','H_k_bits', f'CE_k(w={SAVE_CE_WINDOW})']
        if 'p' in df_new.columns:
            base_cols.insert(2, 'p')
        if 'q' in df_new.columns:
            base_cols.insert(2, 'q')
        final_cols = [c for c in base_cols if c in df_new.columns]
        df_new = df_new[final_cols]

        # Build unique key for rows (Model|Size|q:/p:)
        def _build_key(df):
            keys = []
            for _, r in df.iterrows():
                model = r.get('Model', '')
                size_s = r.get('Size', '')
                if 'q' in df.columns and pd.notna(r.get('q', np.nan)):
                    qp = f"q:{int(r['q'])}"
                elif 'p' in df.columns and pd.notna(r.get('p', np.nan)):
                    qp = f"p:{int(r['p'])}"
                else:
                    qp = 'q:NA'
                keys.append(f"{model}|{size_s}|{qp}")
            return keys

        size_csv = size_dir / 'metrics_summary.csv'
        if size_csv.exists():
            df_old = pd.read_csv(size_csv)
            # Align columns
            all_cols = list({*df_old.columns.tolist(), *df_new.columns.tolist()})
            for c in all_cols:
                if c not in df_old.columns:
                    df_old[c] = np.nan
                if c not in df_new.columns:
                    df_new[c] = np.nan
            df_old['__key__'] = _build_key(df_old)
            df_new['__key__'] = _build_key(df_new)
            df_merged = pd.concat([df_old, df_new], ignore_index=True)
            df_merged = df_merged.drop_duplicates(subset='__key__', keep='last').drop(columns='__key__')
        else:
            df_new['__key__'] = _build_key(df_new)
            df_merged = df_new.drop(columns='__key__')

        # Sort within size by q or p
        sort_key = 'q' if 'q' in df_merged.columns and df_merged['q'].notna().any() else ('p' if 'p' in df_merged.columns else None)
        if sort_key is not None:
            df_merged = df_merged.sort_values(sort_key)

        # Persist and print
        out_cols = [c for c in base_cols if c in df_merged.columns]
        df_merged[out_cols].round(4).to_csv(size_csv, index=False)
        print(df_merged[out_cols].round(4))

        # Incrementally update global CSV after each size
        if all_runs:
            df_all_new = pd.DataFrame(all_runs)
            preferred = ['Model','Size','p','q','L_k','ODR_k','|Delta_c_k|','s_k','H_k_bits', f'CE_k(w={SAVE_CE_WINDOW})']
            all_cols = [c for c in preferred if c in df_all_new.columns]
            df_all_new = df_all_new[all_cols]

            def _build_key(df):
                keys = []
                for _, r in df.iterrows():
                    model = r.get('Model', '')
                    size_s = r.get('Size', '')
                    if 'q' in df.columns and pd.notna(r.get('q', np.nan)):
                        qp = f"q:{int(r['q'])}"
                    elif 'p' in df.columns and pd.notna(r.get('p', np.nan)):
                        qp = f"p:{int(r['p'])}"
                    else:
                        qp = 'q:NA'
                    keys.append(f"{model}|{size_s}|{qp}")
                return keys

            all_csv = BASE_RESULTS_DIR / 'all_metrics_summary.csv'
            if all_csv.exists():
                df_all_old = pd.read_csv(all_csv)
                all_union_cols = list({*df_all_old.columns.tolist(), *df_all_new.columns.tolist()})
                for c in all_union_cols:
                    if c not in df_all_old.columns:
                        df_all_old[c] = np.nan
                    if c not in df_all_new.columns:
                        df_all_new[c] = np.nan
                df_all_old['__key__'] = _build_key(df_all_old)
                df_all_new['__key__'] = _build_key(df_all_new)
                df_all_merged = pd.concat([df_all_old, df_all_new], ignore_index=True)
                df_all_merged = df_all_merged.drop_duplicates(subset='__key__', keep='last').drop(columns='__key__')
            else:
                df_all_new['__key__'] = _build_key(df_all_new)
                df_all_merged = df_all_new.drop(columns='__key__')

            out_cols2 = [c for c in preferred if c in df_all_merged.columns]
            df_all_merged[out_cols2].round(4).to_csv(all_csv, index=False)

# Optionally save all results in one CSV at root (only median metrics)
if all_runs:
    df_all_new = pd.DataFrame(all_runs)
    preferred = ['Model','Size','p','q','L_k','ODR_k','|Delta_c_k|','s_k','H_k_bits', f'CE_k(w={SAVE_CE_WINDOW})']
    all_cols = [c for c in preferred if c in df_all_new.columns]
    df_all_new = df_all_new[all_cols]

    def _build_key(df):
        keys = []
        for _, r in df.iterrows():
            model = r.get('Model', '')
            size_s = r.get('Size', '')
            if 'q' in df.columns and pd.notna(r.get('q', np.nan)):
                qp = f"q:{int(r['q'])}"
            elif 'p' in df.columns and pd.notna(r.get('p', np.nan)):
                qp = f"p:{int(r['p'])}"
            else:
                qp = 'q:NA'
            keys.append(f"{model}|{size_s}|{qp}")
        return keys

    all_csv = BASE_RESULTS_DIR / 'all_metrics_summary.csv'
    if all_csv.exists():
        df_all_old = pd.read_csv(all_csv)
        # Align columns
        all_union_cols = list({*df_all_old.columns.tolist(), *df_all_new.columns.tolist()})
        for c in all_union_cols:
            if c not in df_all_old.columns:
                df_all_old[c] = np.nan
            if c not in df_all_new.columns:
                df_all_new[c] = np.nan
        df_all_old['__key__'] = _build_key(df_all_old)
        df_all_new['__key__'] = _build_key(df_all_new)
        df_all_merged = pd.concat([df_all_old, df_all_new], ignore_index=True)
        df_all_merged = df_all_merged.drop_duplicates(subset='__key__', keep='last').drop(columns='__key__')
    else:
        df_all_new['__key__'] = _build_key(df_all_new)
        df_all_merged = df_all_new.drop(columns='__key__')

    out_cols = [c for c in preferred if c in df_all_merged.columns]
    df_all_merged[out_cols].round(4).to_csv(all_csv, index=False)
    print("\nSaved:", all_csv)


Models to evaluate: ['tcm', 'cheng2020-anchor', 'cheng2020-attn', 'jpeg', 'webp', 'bmshj2018-factorized', 'bmshj2018-hyperprior', 'mbt2018-mean', 'mbt2018']

=== Model: tcm | Size: 256x256 ===
Evaluating p=64...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0280, ODR_k=0.0288, |Δc_k|=0.7587, s_k=10.3361, H_k=0.3645, CE_k(w=2)=0.9761
Evaluating p=128...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0019, ODR_k=0.0019, |Δc_k|=0.0684, s_k=3.5717, H_k=0.0333, CE_k(w=2)=0.9981
  Model     Size    p     L_k   ODR_k  |Delta_c_k|      s_k  H_k_bits  \
0   tcm  256x256   64  0.0280  0.0288       0.7587  10.3361    0.3645   
1   tcm  256x256  128  0.0019  0.0019       0.0684   3.5717    0.0333   

   CE_k(w=2)  
0     0.9761  
1     0.9981  

=== Model: tcm | Size: 512x512 ===
Evaluating p=64...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.0264, ODR_k=0.0271, |Δc_k|=1.1538, s_k=18.5504, H_k=0.3637, CE_k(w=2)=0.9776
Evaluating p=128...
Sum

Downloading: "https://compressai.s3.amazonaws.com/models/v1/bmshj2018-factorized-prior-6-9b02ea3a.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/bmshj2018-factorized-prior-6-9b02ea3a.pth.tar
100%|██████████| 27.3M/27.3M [00:09<00:00, 2.97MB/s]


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.3842, ODR_k=0.6246, |Δc_k|=0.5611, s_k=4.6099, H_k=2.1764, CE_k(w=2)=0.8855
                  Model   Size  q     L_k   ODR_k  |Delta_c_k|     s_k  \
0  bmshj2018-factorized  64x64  1  0.9055  9.6772       2.4135  7.2100   
1  bmshj2018-factorized  64x64  2  0.8541  5.8873       1.5503  6.8523   
2  bmshj2018-factorized  64x64  3  0.8115  4.3146       1.2868  6.2386   
3  bmshj2018-factorized  64x64  4  0.7457  2.9460       0.7765  5.9837   
4  bmshj2018-factorized  64x64  5  0.5704  1.3324       0.8149  5.1637   
5  bmshj2018-factorized  64x64  6  0.3842  0.6246       0.5611  4.6099   

   H_k_bits  CE_k(w=2)  
0    3.4874     0.4416  
1    3.5677     0.6126  
2    3.3291     0.7455  
3    3.1455     0.8036  
4    2.4970     0.8785  
5    2.1764     0.8855  

=== Model: bmshj2018-factorized | Size: 128x128 ===
Evaluating quality 1...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.9608, ODR_k=24.8853, |Δ

Downloading: "https://compressai.s3.amazonaws.com/models/v1/bmshj2018-hyperprior-1-7eb97409.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/bmshj2018-hyperprior-1-7eb97409.pth.tar
100%|██████████| 20.2M/20.2M [00:06<00:00, 3.25MB/s]


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.7417, ODR_k=2.9104, |Δc_k|=0.8110, s_k=5.3040, H_k=3.0882, CE_k(w=2)=0.7951
Evaluating quality 2...


...

Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.3414, ODR_k=0.5188, |Δc_k|=0.5386, s_k=4.6269, H_k=1.9663, CE_k(w=2)=0.8948
Evaluating quality 5...


Downloading: "https://compressai.s3.amazonaws.com/models/v1/mbt2018-5-b4a046dd.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/mbt2018-5-b4a046dd.pth.tar
100%|██████████| 118M/118M [00:23<00:00, 5.18MB/s] 


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.1390, ODR_k=0.1616, |Δc_k|=0.3828, s_k=4.0623, H_k=1.1145, CE_k(w=2)=0.9299
Evaluating quality 6...


Downloading: "https://compressai.s3.amazonaws.com/models/v1/mbt2018-6-7052e5ea.pth.tar" to /home/nkalmykov/.cache/torch/hub/checkpoints/mbt2018-6-7052e5ea.pth.tar
100%|██████████| 118M/118M [00:25<00:00, 4.85MB/s] 


Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.1383, ODR_k=0.1605, |Δc_k|=0.2823, s_k=3.1410, H_k=1.0742, CE_k(w=2)=0.9499
     Model   Size  q     L_k   ODR_k  |Delta_c_k|     s_k  H_k_bits  CE_k(w=2)
0  mbt2018  64x64  1  0.6990  2.4180       1.0654  7.0575    3.3909     0.7735
1  mbt2018  64x64  2  0.5137  1.0632       0.7813  5.7616    2.6576     0.8099
2  mbt2018  64x64  3  0.3730  0.5962       0.8362  5.2888    2.3868     0.8151
3  mbt2018  64x64  4  0.3414  0.5188       0.5386  4.6269    1.9663     0.8948
4  mbt2018  64x64  5  0.1390  0.1616       0.3828  4.0623    1.1145     0.9299
5  mbt2018  64x64  6  0.1383  0.1605       0.2823  3.1410    1.0742     0.9499

=== Model: mbt2018 | Size: 128x128 ===
Evaluating quality 1...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.6927, ODR_k=2.2772, |Δc_k|=2.0669, s_k=11.3931, H_k=3.3423, CE_k(w=2)=0.7861
Evaluating quality 2...
Summary (median over k; H in bits; CE window=2; runs=100): L_k=0.5167, ODR_k